In [1]:

with open("text.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Файл успішно прочитано!")
print("Кількість слів:", len(text.split()))


Файл успішно прочитано!
Кількість слів: 796


In [2]:

%%writefile wer.py

def calculate_wer(reference, hypothesis):
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()

    n = len(ref)
    m = len(hyp)

    # Створюємо таблицю для порівняння слів
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    # Знаходимо мінімальну кількість помилок
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(
                    dp[i - 1][j - 1] + 1,
                    dp[i - 1][j] + 1,
                    dp[i][j - 1] + 1
                )

    # Підраховуємо заміни, видалення та вставки
    s = d = ins = 0
    i, j = n, m

    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1]:
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            s += 1
            i -= 1
            j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            d += 1
            i -= 1
        else:
            ins += 1
            j -= 1

    wer = (s + d + ins) / n if n > 0 else 0

    return {
        "WER": round(wer, 4),
        "S": s,
        "D": d,
        "I": ins,
        "N": n
    }


Writing wer.py


In [3]:

from wer import calculate_wer

reference = "Я сьогодні читаю цікаву книгу"
hypothesis = "Я сьогодні читаю нову книгу"

result = calculate_wer(reference, hypothesis)

print("Результати перевірки:")
print(result)


Результати перевірки:
{'WER': 0.2, 'S': 1, 'D': 0, 'I': 0, 'N': 5}


In [6]:

# Читаємо текст із файлу
with open("text.txt", "r", encoding="utf-8") as file:
    original = file.read()

# Беремо перші 150 слів
words = original.split()[:150]

# Зберігаємо еталонний текст
with open("reference.txt", "w", encoding="utf-8") as file:
    file.write(" ".join(words))

# Створюємо другий текст із кількома змінами
changed = words.copy()

changed[10] = "інформація"   # Заміна слова
del changed[25]              # Видалення слова
changed.insert(40, "сьогодні")  # Додавання слова

# Зберігаємо змінений текст
with open("hypothesis.txt", "w", encoding="utf-8") as file:
    file.write(" ".join(changed))

print("Два тексти успішно створено!")
print("Еталонний текст:", len(words), "слів")
print("Змінений текст:", len(changed), "слів")


Два тексти успішно створено!
Еталонний текст: 150 слів
Змінений текст: 150 слів


In [8]:

import json
from wer import calculate_wer

# Читаємо обидва тексти
with open("reference.txt", "r", encoding="utf-8") as file:
    reference = file.read()

with open("hypothesis.txt", "r", encoding="utf-8") as file:
    hypothesis = file.read()

# Обчислюємо метрики
results = calculate_wer(reference, hypothesis)

# Зберігаємо результати у JSON
with open("results.json", "w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=4)

print("РЕЗУЛЬТАТИ АНАЛІЗУ:")
print(json.dumps(results, ensure_ascii=False, indent=4))
print("Результати збережено у results.json")


РЕЗУЛЬТАТИ АНАЛІЗУ:
{
    "WER": 0.02,
    "S": 1,
    "D": 1,
    "I": 1,
    "N": 150
}
Результати збережено у results.json
